# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path


In [21]:
import pandas as pd
import glob

file_path = glob.glob('/content/*.csv')[0]
df = pd.read_csv(file_path)

print("Loaded file:", file_path)
print(df.head())



Loaded file: /content/co2_emissions.csv
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [ ]:
# Review the countries, regions and emission values
print('Countries in the dataset:')
print(', '.join(sorted(df['Country'].dropna().unique())))

print('\nEmission value range:')
print(f"{df['CO2_Mt'].min():,.1f} to {df['CO2_Mt'].max():,.1f} Mt")

latest_year = df['Year'].max()
latest_data = df[df['Year'] == latest_year]
print(f'\nAverage emissions by region in {latest_year}:')
print(latest_data.groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [22]:
# Keep every Asian country in the comparison
asia_df = df.loc[df['Region'].eq('Asia')].copy()
asia_df = asia_df.sort_values(['Country', 'Year'])

# Select India as the country to highlight
highlight_country = 'India'
if highlight_country not in asia_df['Country'].unique():
    highlight_country = asia_df['Country'].value_counts().index[0]

highlight_data = asia_df[asia_df['Country'] == highlight_country]
first_emission = highlight_data.iloc[0]['CO2_Mt']
last_emission = highlight_data.iloc[-1]['CO2_Mt']
change = last_emission - first_emission
change_pct = (change / first_emission) * 100

fig = go.Figure()

# Draw the remaining Asian countries in the background
for country, country_data in asia_df.groupby('Country'):
    if country != highlight_country:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#EEEEEE', width=1.5),
            hovertemplate=f'{country}<br>Year: %{{x}}<br>CO₂: %{{y:,.1f}} Mt<extra></extra>',
            showlegend=False
        ))

# Add the selected country as the visual focus
fig.add_trace(go.Scatter(
    x=highlight_data['Year'],
    y=highlight_data['CO2_Mt'],
    mode='lines',
    line=dict(color='#0072B2', width=3),
    hovertemplate=f'{highlight_country}<br>Year: %{{x}}<br>CO₂: %{{y:,.1f}} Mt<extra></extra>',
    showlegend=False
))

# Label the highlighted line at its final year
fig.add_annotation(
    x=highlight_data.iloc[-1]['Year'],
    y=last_emission,
    text=highlight_country,
    showarrow=False,
    xshift=32,
    font=dict(size=12, color='#0072B2')
)

fig.update_layout(
    title=dict(
        text=f'{highlight_country}: CO₂ emissions increased by {change_pct:.1f}% from {highlight_data.iloc[0]["Year"]:.0f} to {highlight_data.iloc[-1]["Year"]:.0f}',
        x=0.02
    ),
    template='plotly_white',
    xaxis=dict(
        title='Year',
        showgrid=False,
        showline=False,
        dtick=5
    ),
    yaxis=dict(
        title='CO₂ emissions (Mt)',
        showline=False,
        gridcolor='#EEEEEE',
        zeroline=False
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=65, r=90, t=85, b=55),
    font=dict(family='Arial', size=12)
)

fig.show()
